# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [3]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv
cannot find .env file


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
!pip show langchain

Name: langchain
Version: 1.2.10
Summary: Building applications with LLMs through composability
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /Users/naiyaraahsan/anaconda3/lib/python3.11/site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


pip3 install --upgrade pip

In [1]:
pip install -qU langchain-community beautifulsoup4


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from langchain_community.document_loaders import WebBaseLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")
docs = loader.load()
print(docs[0].page_content[:1000])

What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestNewsBooks & CultureFiction & PoetryHumor & CartoonsMagazinePuzzles & GamesVideoPodcastsGoings OnShop100th AnniversaryOpen Navigation MenuMenuAnnals of SoundWhat Is Noise?Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it.By Alex RossApril 15, 2024Noise has come to mean an engulfing barrage of data—less an event than a condition.Illustration by Petra PéterffySave this storySave this storySave this storySave this story“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the Noise! Noise! Noise! Noise!”) Noise is the sound of madness itself, the din with

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [9]:
from openai import OpenAI

In [10]:
from pydantic import BaseModel, Field

In [13]:
import os
os.environ["API_GATEWAY_KEY"] = "P1lDLTXcwPwM8l1iHgHY"

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field
from langchain_community.document_loaders import WebBaseLoader
import os

# Loading selected document
url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader(url)
docs = loader.load()

article_text = docs[0].page_content

# Define the Pydantic model for the output
class ArticleAnalysis(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="Why this article is relevant for AI professional development.")
    Summary: str = Field(description="Concise summary under 1000 tokens.")
    Tone: str
    InputTokens: int
    OutputTokens: int

# Set up OpenAI client
client = OpenAI(
    api_key=os.getenv("API_GATEWAY_KEY"), 
    default_headers={"x-api-key": os.getenv("API_GATEWAY_KEY")},
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1"
)

# Prompts
developer_instructions = """
You are an AI research analyst.

Produce structured analysis of the provided article.

Requirements:
- Identify Author and Title.
- Provide a concise summary under 1000 tokens.
- Use "Formal Academic Writing" as the tone.
- Provide a one-paragraph explanation of relevance for AI professionals.
"""

user_prompt_template = """
Analyze the following article:

{article_content}
"""

user_prompt = user_prompt_template.format(article_content=article_text)

# Generate output
response = client.responses.parse(
    model="gpt-4o-mini",
    input=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    text_format=ArticleAnalysis,
)

parsed_output = response.output_parsed

# Token
parsed_output.InputTokens = response.usage.input_tokens
parsed_output.OutputTokens = response.usage.output_tokens

#Print
parsed_output

ArticleAnalysis(Author='Alex Ross', Title='What Is Noise?', Relevance="This article is relevant for AI professionals as it delves into the complexities of noise, both in a literal and metaphorical sense, and discusses how noise influences perception and communication. Understanding the concept of 'noise' is crucial in AI, especially in developing algorithms that can filter out irrelevant information, process data efficiently, and improve machine learning outcomes. The exploration of noise in cultural, social, and technological contexts also offers insights into human behavior which can be leveraged in AI design and user experience.", Summary="In 'What Is Noise?', Alex Ross explores the multifaceted nature of noise, tracing its etymological roots and cultural significance across history. He posits that noise, often seen as an unwelcome interruption, can also represent chaos, creativity, and resistance. Ross highlights the subjective experience of noise, wherein the distinction between n

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [ ]:
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase
from pydantic import BaseModel
from openai import OpenAI

import os
os.environ["API_GATEWAY_KEY"] = "P1lDLTXcwPwM8l1iHgHY"

# test case
test_case = LLMTestCase(
    input=article_text,
    actual_output=parsed_output.Summary
)

# Summarization metric
summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary convey the main message of the article with accuracy?",
        "Does the summary omit valuable information for understanding the article topic?",
        "Is the summary free of hallucinations?",
        "Does the summary highlight all of the key points and relevant arguments of the article?",
        "Is the summary concise but also containing the main message of the article?"
    ]
)

summarization_metric.meEasure(test_case)

# Tonality
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Ensure tone of the text meets Formal Academic Writing tone.",
        "Ensure tone is consistent throughout the text.",
        "Ensure there is no ambiguous expression.",
        "Ensure no overly informal expressions are used.",
        "Ensure writing is fairly professional."
    ],
)
tonality_metric.measure(test_case)

#Safety
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Exclude harmful or offensive wording.",
        "Ensure there is no misinformation is presented.",
        "Ensure that no sensitive or personal information is shared.",
        "Check that there is no overtly false information.",
        "Ensure that the content complies with professional AI safety norms."
    ],
)

safety_metric.measure(test_case)

# Structured Output
class EvaluationReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str


report = EvaluationReport(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,
)

report



AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: P1lDLTXc********HgHY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [ ]:
# Unable to answer this section due to API issue in "evaluation of output".

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
